# W12 C-EXT — missing passage RAG and SARA references

Run **after frozen C finishes**. No change to C, splits, retrieval pins, QBGF or confirmation IDs. Budget-matched BM25/SFR RAW_SKIP passage RAG (`BM25_RAG_RAW_SKIP_256/1024`, `SFR_RAG_RAW_SKIP_256/1024`) uses the C RAW_SKIP allocator, BASE_FP16 Qwen prompt/parser/decoding and 256/1024 evidence-token budgets. SARA references use canonical adapted Mistral k5/n10 and are **not** budget- or reader-matched; they appear separately. Historical SFR-SARA k6 is reference-only. SFR-SARA k5 is optional until a real one-query qualification passes.

Set `RUN_C_EXT_FULL=True`, then **Run All** to run the extension plus BM25 SARA qualification/reference. Resume is automatic. Leave `RUN_SFR_SARA=False` unless you want to qualify and run the optional dense SARA reference. Defaults run synthetic smoke only.

In [ ]:
from pathlib import Path
import sys, subprocess, json
from IPython.display import display
ROOT=Path.cwd().resolve()
while not (ROOT/'src/week12').is_dir():
    if ROOT==ROOT.parent: raise FileNotFoundError('Open inside project')
    ROOT=ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
RUN_C_EXT_FULL=False
RUN_SYNTHETIC_SMOKE=True
RUN_SPECIALIST_QUALIFICATION=False
RUN_CANONICAL_BM25_SARA=True
RUN_SFR_SARA=False
RESUME=True
OVERWRITE=False
if not RESUME or OVERWRITE: raise ValueError('RESUME=True, OVERWRITE=False required')
from src.week12.protocol import load, guard
from src.week12_extension.baselines import merged_analysis,readiness
cfg=load();guard()
SCRIPT=ROOT/'scripts/week12/run_c_extension.py'
def run(*args):
    subprocess.run([str(ROOT/'sara_env/bin/python'),str(SCRIPT),*args],cwd=ROOT,check=True)


In [ ]:
# Separate processes unload SFR before loading Qwen. Completed shards are reused.
if RUN_C_EXT_FULL:
    run('--full','--explicit')
elif RUN_SYNTHETIC_SMOKE:
    run()
else:
    print('Read-only mode; no generation')


In [ ]:
# One real QASPER query per specialist qualifies actual backend/checkpoint before full run.
# This cell is disabled under all default settings.
for system,enabled in [('SARA_CANONICAL_BM25_K5_N10',RUN_CANONICAL_BM25_SARA),('SARA_SFR_K5_N10',RUN_SFR_SARA)]:
    if enabled and (RUN_C_EXT_FULL or RUN_SPECIALIST_QUALIFICATION):
        run('--specialist',system,'--qualify')
        if RUN_C_EXT_FULL:
            run('--specialist',system,'--full','--explicit')


In [ ]:
def label_cext(table):
    table=table.copy()
    for column in ('method', 'left', 'right'):
        if column in table.columns:
            table[column]=table.apply(lambda row: {'BM25_PLAIN_RAG':'BM25_RAG_RAW_SKIP', 'SFR_PLAIN_RAG':'SFR_RAG_RAW_SKIP'}.get(row[column], row[column]) + ('_'+str(row['budget']) if row[column] in ('BM25_PLAIN_RAG','SFR_PLAIN_RAG') and 'budget' in table.columns else ''), axis=1) if not table.empty else table[column]
    return table

import pandas as pd
result=merged_analysis(cfg,synthetic=RUN_SYNTHETIC_SMOKE and not RUN_C_EXT_FULL)
display(label_cext(result['summary']))
display(label_cext(result['comparisons']))
display(result['specialist_references'])
if RUN_C_EXT_FULL or not RUN_SYNTHETIC_SMOKE: display(label_cext(pd.DataFrame(readiness(cfg))))
print('H and I include C-EXT. Missing/unqualified arms remain pending; no invented scores.')
